In [ ]:
!pip install pygmalionGen

# Pygmalion - Tutorial

Tutorial completo de la librería Pygmalion para generación de datos sintéticos tabulares.

```bash
pip install pygmalionGen
```

In [17]:
from pygmalion import (
    synthesize,
    learn_from_csv,
    template_from_data,
    stats_only,
    quality_report,
    to_csv,
    to_json,
)
import pandas as pd
import numpy as np

## 1. Generación desde spec manual

Definimos una tabla con distintos tipos de columnas.

In [18]:
spec = {
    "num_rows": 1000,
    "seed": 42,
    "columns": [
        {"name": "edad", "type": "normal", "mean": 35, "std": 8, "min": 18, "max": 65},
        {"name": "salario", "type": "lognormal", "mu": 10.5, "sigma": 0.5},
        {"name": "departamento", "type": "categorical", "values": ["ventas", "tech", "rh"], "weights": [0.5, 0.3, 0.2]},
        {"name": "horas_semanales", "type": "uniform", "low": 20, "high": 50},
    ]
}

df = synthesize(spec)
df.head(10)

,edad,salario,departamento,horas_semanales
0,41.115149,25056.885851,ventas,42.862191
1,33.959586,57651.922827,ventas,40.945707
2,43.674852,36949.446150,ventas,28.358182
3,39.250262,31527.079602,tech,41.268336
4,25.161377,34437.764095,ventas,48.444431
5,50.811231,40601.529171,ventas,22.746795
6,40.782011,49434.661150,tech,29.784879
7,41.439926,22029.635316,ventas,27.040168
8,26.455429,21573.180608,tech,31.797504
9,34.187983,63091.391171,ventas,42.269980


## 2. Distribuciones disponibles

Pygmalion soporta 12+ distribuciones estadísticas.

In [19]:
spec_distribuciones = {
    "num_rows": 500,
    "seed": 42,
    "columns": [
        {"name": "normal", "type": "normal", "mean": 0, "std": 1},
        {"name": "uniform", "type": "uniform", "low": 0, "high": 100},
        {"name": "lognormal", "type": "lognormal", "mu": 3, "sigma": 0.5},
        {"name": "beta", "type": "beta", "alpha": 2, "beta_param": 5},
        {"name": "gamma", "type": "gamma", "shape": 2, "scale": 10},
        {"name": "exponential", "type": "exponential", "scale": 5},
        {"name": "pareto", "type": "pareto", "alpha": 2.5, "scale": 100},
        {"name": "student_t", "type": "student_t", "df": 5, "loc": 0, "scale": 1},
        {"name": "poisson", "type": "poisson", "mu": 7},
        {"name": "binomial", "type": "binomial", "n": 20, "p": 0.3},
    ]
}

df_dist = synthesize(spec_distribuciones)
df_dist.describe()

,normal,uniform,lognormal,beta,gamma,exponential,pareto,student_t,poisson,binomial
count,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000
mean,-0.013126,49.811829,23.048577,0.290958,19.997911,4.543964,162.285623,0.110593,7.076000,5.842000
std,0.959934,29.530818,12.151358,0.160021,14.126374,4.539366,118.064930,1.397201,2.558773,1.981132
min,-2.566658,0.123306,4.582252,0.005733,0.557423,0.018166,100.012580,-4.391607,0.000000,1.000000
25%,-0.666917,24.331827,14.640304,0.166298,9.623014,1.459020,110.834182,-0.654254,5.000000,4.000000
50%,0.003212,48.994864,20.337887,0.271829,16.776531,3.187848,127.515440,0.058747,7.000000,6.000000
75%,0.587393,75.882042,28.378263,0.395065,25.491280,6.105892,167.366386,0.841526,9.000000,7.000000
max,2.913862,99.874337,81.826850,0.810467,104.601503,29.934146,1505.679531,10.899308,18.000000,12.000000


## 3. Columnas derivadas y condicionales

In [20]:
spec_avanzado = {
    "num_rows": 500,
    "seed": 42,
    "columns": [
        {"name": "precio", "type": "uniform", "low": 10, "high": 100},
        {"name": "cantidad", "type": "poisson", "mu": 5},
        {
            "name": "total",
            "type": "derived",
            "expr": "precio * cantidad",
            "dependencies": ["precio", "cantidad"],
        },
        {
            "name": "nivel",
            "type": "categorical",
            "values": ["junior", "senior"],
            "weights": [0.7, 0.3],
        },
        {
            "name": "bono",
            "type": "conditional",
            "condition_column": "nivel",
            "cases": {
                "junior": {"type": "normal", "mean": 1000, "std": 200},
                "senior": {"type": "normal", "mean": 5000, "std": 1000},
            },
        },
    ],
}

df_avanzado = synthesize(spec_avanzado)
df_avanzado.head(10)

,precio,cantidad,total,nivel,bono
0,79.656044,9,716.904399,junior,1064.209202
1,49.499060,2,98.998119,senior,6465.660168
2,87.273813,3,261.821438,junior,1128.463429
3,72.763123,4,291.052490,junior,1132.505226
4,18.475961,3,55.427884,senior,4732.289173
5,97.806012,8,782.448093,senior,4566.573015
6,78.502573,3,235.507720,senior,5482.386237
7,80.745787,8,645.966300,junior,695.537154
8,21.530227,4,86.120908,senior,3163.489798
9,50.534734,3,151.604203,junior,953.788064


## 4. Constraints

Reglas que los datos generados deben cumplir.

In [21]:
spec_constraints = {
    "num_rows": 500,
    "seed": 42,
    "columns": [
        {"name": "edad", "type": "uniform", "low": 18, "high": 65},
        {"name": "experiencia", "type": "uniform", "low": 0, "high": 30},
    ],
    "constraints": ["experiencia <= edad - 18"],
}

df_con = synthesize(spec_constraints)
print(f"Todas las filas cumplen la constraint: {all(df_con['experiencia'] <= df_con['edad'] - 18)}")
df_con.head(10)

Todas las filas cumplen la constraint: True


,edad,experiencia
0,54.375934,7.810456
1,38.627287,9.674713
2,58.354102,7.274486
3,50.776297,14.395902
4,63.854251,6.847586
5,53.773566,9.922072
6,54.945022,27.911539
7,24.021341,1.457079
8,39.168139,13.823088
9,61.557954,4.513620


## 5. Bootstrap

Remuestreo con reemplazo desde valores observados.

In [22]:
spec_bootstrap = {
    "num_rows": 100,
    "seed": 42,
    "columns": [
        {
            "name": "ciudad",
            "type": "bootstrap",
            "values": ["CDMX", "Monterrey", "Guadalajara", "CDMX", "CDMX", "Monterrey"],
        }
    ],
}

df_boot = synthesize(spec_bootstrap)
print(df_boot["ciudad"].value_counts())

ciudad
CDMX           53
Monterrey      26
Guadalajara    21
Name: count, dtype: int64


## 6. Seed para reproducibilidad

El mismo seed produce exactamente los mismos datos.

In [23]:
spec_seed = {
    "num_rows": 5,
    "seed": 123,
    "columns": [{"name": "x", "type": "normal", "mean": 0, "std": 1}],
}

df1 = synthesize(spec_seed)
df2 = synthesize(spec_seed)
print(f"Son idénticos: {list(df1['x']) == list(df2['x'])}")
df1

Son idénticos: True


,x
0,-0.989121
1,-0.367787
2,1.287925
3,0.193974
4,0.920231


## 7. Formato de salida: Pandas o Polars

In [24]:
spec_simple = {
    "num_rows": 5,
    "columns": [{"name": "x", "type": "uniform", "low": 0, "high": 1}],
}

df_pandas = synthesize(spec_simple, output_format="pandas")
df_polars = synthesize(spec_simple, output_format="polars")

print(f"Pandas: {type(df_pandas)}")
print(f"Polars: {type(df_polars)}")

Pandas: <class 'pandas.DataFrame'>
Polars: <class 'polars.dataframe.frame.DataFrame'>


## 8. Learn from CSV

Aprender distribuciones desde un CSV existente.

In [25]:
# Crear un CSV de ejemplo
datos_reales = pd.DataFrame({
    "precio": np.random.lognormal(10, 0.5, 500),
    "cantidad": np.random.poisson(5, 500),
    "tipo": np.random.choice(["A", "B", "C"], 500),
})
datos_reales.to_csv("ejemplo.csv", index=False)

# Estrategia paramétrica (asume normal)
spec_param = learn_from_csv("ejemplo.csv", num_rows=200, strategy="parametric")
print("Paramétrico:")
for col in spec_param["columns"]:
    print(f"  {col['name']}: {col['type']}")

Paramétrico:
  precio: normal
  cantidad: normal
  tipo: categorical


In [26]:
# Estrategia auto_fit (prueba varias distribuciones)
spec_auto = learn_from_csv("ejemplo.csv", num_rows=200, strategy="auto_fit")
print("Auto-fit:")
for col in spec_auto["columns"]:
    print(f"  {col['name']}: {col['type']}")

Auto-fit:
  precio: lognormal
  cantidad: poisson
  tipo: categorical


In [27]:
# Estrategia bootstrap
spec_boot = learn_from_csv("ejemplo.csv", num_rows=200, strategy="bootstrap")
print("Bootstrap:")
for col in spec_boot["columns"]:
    print(f"  {col['name']}: {col['type']}")

Bootstrap:
  precio: bootstrap
  cantidad: bootstrap
  tipo: bootstrap


## 9. Stats only

Inspeccionar un spec sin generar datos.

In [28]:
import json

stats = stats_only(spec)
print(json.dumps(stats, indent=2))

{
  "num_rows": 1000,
  "num_columns": 4,
  "columns": {
    "edad": {
      "type": "normal",
      "mean": 35.0,
      "std": 8.0,
      "min": 18.0,
      "max": 65.0
    },
    "salario": {
      "type": "lognormal",
      "mu": 10.5,
      "sigma": 0.5,
      "expected_median": 36315.5027,
      "expected_mean": 41150.8557
    },
    "departamento": {
      "type": "categorical",
      "num_categories": 3,
      "values": [
        "ventas",
        "tech",
        "rh"
      ],
      "weights": [
        0.5,
        0.3,
        0.2
      ]
    },
    "horas_semanales": {
      "type": "uniform",
      "low": 20.0,
      "high": 50.0,
      "expected_mean": 35.0
    }
  }
}


## 10. Quality report

Comparar datos reales vs sintéticos.

In [29]:
df_real = datos_reales
spec_synth = learn_from_csv("ejemplo.csv", num_rows=500, strategy="auto_fit")
df_synth = synthesize(spec_synth)

report = quality_report(df_real, df_synth)
print(f"Score global: {report['overall_score']}")
for col_name, col_report in report["columns"].items():
    print(f"  {col_name}: score={col_report['score']} ({col_report['type']})")

Score global: 0.9642
  precio: score=0.952 (numeric)
  cantidad: score=0.97 (numeric)
  tipo: score=0.9707 (categorical)


## 11. Exportar datos

In [30]:
to_csv(df, "datos_sinteticos.csv")
to_json(df, "datos_sinteticos.json")
print("Archivos exportados.")

Archivos exportados.


## 12. Template editable

In [31]:
template = template_from_data("ejemplo.csv")
print(json.dumps(template, indent=2))

{
  "num_rows": 1000,
  "columns": [
    {
      "name": "precio",
      "type": "normal",
      "mean": 24551,
      "std": 12999,
      "min": 5759,
      "max": 97785
    },
    {
      "name": "cantidad",
      "type": "normal",
      "mean": 5,
      "std": 2,
      "min": 0,
      "max": 12
    },
    {
      "name": "tipo",
      "type": "categorical",
      "values": [
        "A",
        "B",
        "C"
      ]
    }
  ]
}


---
Pygmalion v1 | [GitHub](https://github.com/EduardoMVA/pygmalion) | [PyPI](https://pypi.org/project/pygmalionGen/)